# Speech Analytics: From a Pilot's Snag to the Likely Part

**Speech Analytics &middot; Session 4 of 4** &mdash; Turn a pilot's spoken snag into the most likely faulty part &mdash; with nobody typing

## How to use this notebook

Same as every notebook today: you only **run** the cells, top to bottom, and read what comes out. You do not write any code.

- To run a cell: click it, then press **Shift + Enter** (or the &#9654; play button on its left).
- Run them **in order** &mdash; each cell builds on the one above it.
- If anything looks stuck, use the menu: **Runtime &rarr; Restart and run all**, then start again.

This notebook needs a one-time setup &mdash; the very first code cell below installs two extra speech tools and takes about a minute. After that, everything runs as usual.

As you scroll you will meet four little callouts: 🟦 a new word, 🟩 what a cell just did, 🟧 something to try, 🟪 the recap.

> 💡 **THE BIG IDEA**
>
> All day long we worked with messy data: first written <b>text</b> (the maintenance notes), then a machine's <b>sound</b>. There is one last kind of messy data left: <b>spoken words</b>.<br><br>A pilot reporting a snag would rather <b>speak</b> it than stop and type. So the question becomes: can the computer <b>listen</b> to that snag and help straight away? Yes &mdash; in two moves. First it <b>transcribes</b> the speech into text. Then it runs the <b>same text analytics</b> you already saw in the text sessions to suggest the <b>likely part</b>.<br><br><b>Speech analytics = speech-to-text + text analytics.</b> Nothing new to fear &mdash; it is two familiar pieces snapped together.

> 📍 **THE SCENARIO**
>
> <b>Our scenario.</b> A pilot reports a snag over the radio and says:<br><br><i>"Static on the radio and the signal drops when the aircraft shakes."</i><br><br>Nobody types anything. We want the computer to take that <b>audio</b>, figure out the words, and suggest the <b>most likely part</b> behind the snag &mdash; the antenna, the coax cable, the RF filter (HPF), or the radio box (LRU). Unlike sorting into a team, a snag rarely points at one part for certain &mdash; so the honest answer is a <b>ranked shortlist</b>. Here is the whole pipeline we are about to build:

**The whole pipeline, end to end:**

> 🎙️ **Speak** &rarr; 📝 **Transcribe (ASR)** &rarr; 🔧 **Suggest the likely part**

- **Speak** &mdash; a pilot says the snag out loud: *"Static on the radio and the signal drops when the aircraft shakes."*
- **Transcribe** &mdash; the computer listens and turns the audio into plain **text** (speech-to-text / ASR).
- **Suggest the part** &mdash; the **same text analytics** from the text sessions ranks the likely parts &rarr; top suggestion **Cable (CBL-221)**, with the others close behind.

## Step 1 · One-time setup (this is the slow cell)

> 🟦 **VOCABULARY — Library (a quick reminder)**
>
> A <b>library</b> is a ready-made toolbox someone else wrote so we do not start from scratch. This notebook borrows three: <code>gTTS</code> to <b>make</b> a demo audio clip, <code>openai-whisper</code> to <b>listen</b> to audio and turn it into text, and <code>sentence-transformers</code> to turn a sentence into numbers that capture its <b>meaning</b> (an embedding, just like Session 2).

> ℹ️ **NOTE**
>
> This first cell is the <b>only</b> heavy setup in the whole day, and it takes about a minute as Colab downloads the tools. A wall of install text scrolling by is completely normal &mdash; wait for it to finish (the &#9654; spinner stops) before running the next cell. (A small embedding model, about 80&nbsp;MB, downloads the first time we use it in Step&nbsp;4.)

In [13]:
# One-time install for Colab. The -q flag just means 'quiet' (less text).
# This downloads the extra tools we need. Give it about a minute.
!pip install -q gTTS openai-whisper sentence-transformers

print('Setup finished. The speech and language tools are installed and ready.')

Setup finished. The speech and language tools are installed and ready.


> 🟩 **WHAT JUST HAPPENED**
>
> Colab just downloaded our two speech tools. From here on it behaves like every other notebook today &mdash; run cells in order and read the output. That slow step is over.

## Step 2 · Make a demo spoken report (no microphone needed)

To keep this notebook fully self-contained &mdash; no microphones, no uploading files &mdash; we will *manufacture* a spoken snag from a written sentence. The computer reads our sentence aloud and saves it as an audio file. That file then stands in for a real pilot's recording.

> 🟦 **VOCABULARY — Text-to-speech (TTS)**
>
> <b>Text-to-speech</b> turns written words into spoken audio &mdash; the opposite of what we really care about today. We use it for one small reason only: to <b>fabricate a demo clip</b> so everyone has an identical recording to work with, with no microphone required. The real star of the notebook is the step <i>after</i> this.

In [14]:
from gtts import gTTS                 # text-to-speech tool (makes our demo clip)
from IPython.display import Audio     # lets us play sound inside the notebook

# This written sentence is what we will pretend a pilot spoke as a snag.
spoken_report = 'Static on the radio and the signal drops when the aircraft shakes.'

# Turn that sentence into speech and save it as an audio file named report.mp3
gTTS(spoken_report).save('report.mp3')

print('Created an audio file: report.mp3')
print('It is the spoken version of:')
print('   "' + spoken_report + '"')

Created an audio file: report.mp3
It is the spoken version of:
   "Static on the radio and the signal drops when the aircraft shakes."


In [15]:
# Press play on the little audio bar below to HEAR our demo report.
Audio('report.mp3')

> 🟩 **WHAT JUST HAPPENED**
>
> We just created <code>report.mp3</code> and played it. That audio file is now our <b>stand-in for a real spoken snag</b> &mdash; exactly the kind of clip a pilot's radio call or handheld would produce. From the computer's point of view, it has no idea what words are inside; it only has sound. Our job next is to recover the words.

## Step 3 · Listen to the audio and turn it into text

This is the genuinely new skill of the notebook: handing raw audio to the computer and getting back the words. We load a small listening model and ask it to transcribe our clip.

> 🟦 **VOCABULARY — Speech-to-text / ASR (automatic speech recognition)**
>
> <b>Speech-to-text</b> &mdash; also called <b>ASR</b>, automatic speech recognition &mdash; is a model that <b>listens</b> to audio and writes down the words it hears. It is exactly what your phone does when you dictate a message. We use a model called <b>Whisper</b>, in its smallest <code>tiny</code> size so it runs quickly here.

> ℹ️ **NOTE**
>
> The first time you run the next cell, Whisper downloads its <code>tiny</code> model (a few seconds). It then prints a small warning about running on CPU &mdash; that is harmless and expected on Colab. Just wait for the text.

In [16]:
import whisper

# Load the smallest Whisper model. 'tiny' is fast and fine for short, clear clips.
model = whisper.load_model('tiny')

# Ask it to listen to our audio file and write down the words.
result = model.transcribe('report.mp3')
transcript = result['text'].strip()    # the recovered text

print('What the computer HEARD (the transcript):')
print('   "' + transcript + '"')
print()
print('The original sentence we spoke:')
print('   "' + spoken_report + '"')

/Users/aarshdesai/miniconda3/lib/python3.12/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


What the computer HEARD (the transcript):
   "Static on the radio, and the signal drops when the aircraft shakes."

The original sentence we spoke:
   "Static on the radio and the signal drops when the aircraft shakes."


> 🟩 **WHAT JUST HAPPENED**
>
> The computer <b>listened to sound and produced text</b> &mdash; and it is (nearly) word-for-word the sentence we spoke. It was never told the words; it worked them out from the audio alone. That recovered <code>transcript</code> is now ordinary text, which means we can feed it straight into the same text analytics from the text sessions.

> ⚠️ **WATCH OUT**
>
> ASR is not perfect. With a clear voice and no background noise &mdash; like our demo &mdash; it is excellent. But strong accents, cockpit and radio noise, mumbling, or unusual part names can make it <b>mis-hear</b> a word (a part code like "CBL-221" can come out as "CBL-227"). In the real world you check important transcripts, and a noisy clip will read less cleanly than this quiet studio-style one. Honest expectations beat magical ones.

## How honest is that transcript? Measure it with WER

> 🟦 **VOCABULARY — Word Error Rate (WER)**
>
> How do we put a number on how well speech-to-text did? <b>WER</b> compares the transcript with a correct reference and counts three kinds of slip &mdash; <b>wrong</b> words (substitutions), <b>missing</b> words (deletions) and <b>extra</b> words (insertions) &mdash; divided by the number of reference words. 0% is perfect. It is simply the speech version of the <i>accuracy</i> idea from the earlier sessions.

In [17]:
# A tiny, self-contained WER (word-level edit distance). No model or internet needed --
# we compare a known-correct sentence against what an ASR system might have heard.
def wer(reference, heard):
    r, h = reference.lower().split(), heard.lower().split()
    d = [[0]*(len(h)+1) for _ in range(len(r)+1)]
    for i in range(len(r)+1): d[i][0] = i
    for j in range(len(h)+1): d[0][j] = j
    for i in range(1, len(r)+1):
        for j in range(1, len(h)+1):
            cost = 0 if r[i-1] == h[j-1] else 1
            d[i][j] = min(d[i-1][j] + 1, d[i][j-1] + 1, d[i-1][j-1] + cost)
    return d[-1][-1] / max(1, len(r))

# A maintenance read-back of a part number -- three ways an ASR might hear it.
reference = 'replace the coax cable part number cbl two two one on com one'
heard = {
    'clean studio voice':       'replace the coax cable part number cbl two two one on com one',
    'some shop-floor noise':    'replace coax cable part number cbl two two one on the com one',
    'misheard the part number': 'replace the coax cable part number cbl two two seven on com one',
}
print(f'{"scenario":26s} {"WER":>6}   part number correct?')
for name, said in heard.items():
    part_ok = 'two two one' in said.lower()
    print(f'{name:26s} {wer(reference, said)*100:5.1f}%   {"yes" if part_ok else "NO  <-- wrong part! (CBL-221 -> CBL-227)"}')

scenario                      WER   part number correct?
clean studio voice           0.0%   yes
some shop-floor noise       15.4%   yes
misheard the part number     7.7%   NO  <-- wrong part! (CBL-221 -> CBL-227)


> 🟩 **WHAT JUST HAPPENED**
>
> Read the three lines. The noisy transcript scores a <b>worse</b> WER (~15%) but every important word survived &mdash; harmless. The third scores a <b>better</b> WER (~8%) yet quietly turned <b>CBL-221</b> into <b>CBL-227</b> &mdash; the one word that matters, now wrong. <b>A low error rate does not mean the transcript is safe.</b>

> ⚠️ **WATCH OUT**
>
> WER weighs every word equally &mdash; mishearing "the" counts the same as mishearing a part number, a callsign or a torque value. So for anything safety- or traceability-critical the rule is simple and non-negotiable: a human verifies the numbers and serials. Speech-to-text is a fast first draft, not the final record.

## When you only need a few words: keyword spotting

> 🟦 **VOCABULARY — Keyword spotting**
>
> Often you do not need a full transcript &mdash; only to know whether one of a few <b>commands</b> was spoken ("start log", "flag defect"). <b>Keyword spotting</b> listens for just a small fixed vocabulary. Because the job is so much narrower, it is far more <b>robust to noise and accents</b> than open-ended transcription &mdash; the right tool when reliability matters more than detail.

In [18]:
# Listen for a tiny fixed command vocabulary inside messy, real-world transcripts.
COMMANDS = ['start log', 'flag defect', 'pressure low', 'engine fire']

noisy_transcripts = [
    'uh start log please',                   # command buried in filler
    'the the flag defect on number two',     # stutter + extra words
    'cabin pressure low warning came on',    # command inside a sentence
    'everything looks normal up here today',  # no command -- just chatter
]
for t in noisy_transcripts:
    hits = [c for c in COMMANDS if c in t.lower()]
    print(f'{t:42s} -> {hits if hits else "(no command detected)"}')

uh start log please                        -> ['start log']
the the flag defect on number two          -> ['flag defect']
cabin pressure low warning came on         -> ['pressure low']
replace the oil filter next service        -> (no command detected)


> 🟩 **WHAT JUST HAPPENED**
>
> Even with stutters, filler words and whole surrounding sentences, the small vocabulary is spotted reliably &mdash; and the line with no command is correctly left alone. This is why hands-free shop-floor tools lean on a handful of wake-words rather than full dictation: a 10-word target is dramatically easier to get right than every word a person might say.

> ℹ️ **NOTE**
>
> <b>Hard by default, strong with domain data.</b> Out of the box, ASR struggles with accents, heavy jargon and radio noise. But it can be <b>trained on a specific domain</b>: the EU <b>HAAWAII</b> project built speech recognition for air-traffic control that reached over <b>95%</b> word recognition for controllers and over <b>90%</b> for pilots after domain training, and about <b>97%</b> on aircraft callsigns once the audio was combined with radar context. Open ATC speech datasets &mdash; <b>ATCOSIM</b> (clean) and <b>ATCO2</b> (noisy) &mdash; are where teams practise exactly this.

## Step 4 · Teach the computer the parts (text analytics)

Now we rebuild the text-analytics idea from the text sessions, self-contained right here. We give the computer a small set of past pilot snags, each already labelled with the part that turned out to be at fault. From those examples it learns which words lean toward which part. This is the same learn-from-past-examples pattern as every notebook today &mdash; only the data is words instead of numbers.

> 🟦 **VOCABULARY — Turning words into numbers (embeddings)**
>
> A model can only do maths on numbers, so first we turn each snag into a row of numbers. This time we use an <b>embedding model</b> &mdash; the same <b>embedding</b> idea from Session 2. It is a small, pre-trained model that reads a sentence and places it as a point in a kind of "meaning space", so snags that <i>mean</i> similar things land near each other &mdash; <b>even when they share no words</b> ("crackly" sits near "static"). That is why it can weigh up a snag worded in a way it has never seen before &mdash; a step up from plain word-counting.

In [19]:
# A small set of PAST pilot snags, each labelled with the PART that turned out to
# be at fault. In real life these would be thousands of historical snags from the logs.
# The parts form a comms / RF chain, and the symptoms deliberately OVERLAP -- so a
# single snag rarely points at one part with certainty.
snags = [
    # --- Antenna (reception, weak/poor signal, worse in weather, physical) ---
    'no vhf reception and the signal drops out completely in flight',
    'comms are weak and reception gets worse in heavy rain',
    'the antenna looks cracked and reception is poor on com one',
    'intermittent reception that improves when we change heading',
    'very weak signal strength on the navigation receiver',
    'poor reception and the antenna mount looks corroded',
    'signal fades out at range and comes back when we are closer',
    'reception dropped right after a bird strike near the antenna',
    # --- Cable (intermittent, connector, vibration, static, coax) ---
    'the signal cuts in and out when the aircraft vibrates',
    'static and noise on the radio that changes when you move the connector',
    'intermittent comms and a loose connector behind the panel',
    'reception is noisy and the coax connector is corroded',
    'the signal drops when we wiggle the cable at the back of the radio',
    'crackling noise on transmit and a chafed cable run',
    'an intermittent fault that clears when the harness is flexed',
    'high standing wave ratio and a damaged section of coax',
    # --- HPF (the RF filter: interference, spurious noise, cross-talk, blocking) ---
    'interference from the other transmitter bleeds into com one',
    'spurious noise on the radio whenever the radar is on',
    'strong out of band signals are desensing the receiver',
    'the filter unit is overheating in the avionics bay',
    'cross talk between systems and a failed filter module',
    'unwanted signals getting through and blocking the reception',
    'the high power filter shows a fault and interference is high',
    'radar pulses are leaking into the comms after the filter failed',
    # --- LRU (the radio box: dead unit, BITE fault, no power, fails self-test) ---
    'the radio is completely dead with no power to the unit',
    'a bite fault flag on the transceiver and it will not transmit',
    'the comms box reset itself and now shows a unit failure',
    'no power at the line replaceable unit and the display is blank',
    'the unit fails its self test and will not power up',
    'the transceiver locks up and needs the lru replaced',
    'an internal fault on the radio box, it transmits but does not receive',
    'the line replaceable unit is showing a hardware fault code',
]
parts = (
    ['Antenna'] * 8 +   # first 8 snags -> Antenna
    ['Cable']   * 8 +   # next 8 -> Cable
    ['HPF']     * 8 +   # next 8 -> HPF (the RF filter)
    ['LRU']     * 8     # last 8 -> LRU (the radio box)
)

# Each part has a number engineers actually order against.
PART_NUMBERS = {
    'Antenna': 'ANT-4710',
    'Cable':   'CBL-221',
    'HPF':     'HPF-2200',
    'LRU':     'LRU-118',
}

print('We have', len(snags), 'past pilot snags across', len(set(parts)), 'parts:')
print('   Antenna, Cable, HPF, LRU  (8 examples each)')

We have 32 past pilot snags across 4 parts:
   Antenna, Cable, HPF, LRU  (8 examples each)


> 🟩 **WHAT JUST HAPPENED**
>
> There is our labelled history: 32 short snags, each tagged with the part that was at fault. Eight examples per part. Small, but enough to show the idea. The computer has not learned anything yet &mdash; we have only laid out the examples.

In [20]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression

# Load a small embedding model (it downloads once, about 80 MB). It turns any sentence
# into a row of numbers that captures its MEANING -- the same embedding idea from Session 2.
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Turn every past snag into its embedding, then learn which embeddings go with which part.
X = embedder.encode(snags)
classifier = LogisticRegression(max_iter=1000)
classifier.fit(X, parts)

print('Training done. The computer has learned which snags point to which part.')

/Users/aarshdesai/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7809.07it/s]


Training done. The computer has learned which snags point to which part.


> 🟩 **WHAT JUST HAPPENED**
>
> One <code>.fit()</code> call &mdash; the same word you saw in the earlier sessions &mdash; and the model has learned which words lean toward the Antenna, the Cable, the HPF filter, and the LRU. It is ready to weigh up a brand-new snag.

## Step 5 · Suggest the likely part

Here is the payoff. We take the **transcript** from Step 3 &mdash; the text the computer recovered from the audio &mdash; and ask the trained model which part the snag points to. Speech became text; now text becomes a ranked shortlist of likely parts.

In [21]:
# Use the TRANSCRIBED text from Step 3. (If you skipped the speech cells, fall
# back to the sentence we spoke, so this cell still runs on its own.)
transcript = transcript if 'transcript' in dir() else \
    'static on the radio and the signal drops when the aircraft shakes'

# Turn the snag into its embedding, then ask the model which PART it points to.
v = embedder.encode([transcript])
likely_part = classifier.predict(v)[0]

# How sure is it? Show the probability for every part -- a ranked shortlist.
probs = classifier.predict_proba(v)[0]
labels = classifier.classes_

print('Spoken snag (transcribed):')
print('   "' + transcript + '"')
print()
print('Most likely part:', likely_part, '(P/N ' + PART_NUMBERS[likely_part] + ')')
print()
print('Ranked shortlist (how sure, per part):')
for part, p in sorted(zip(labels, probs), key=lambda kv: -kv[1]):
    bar = '#' * int(round(p * 30))    # a little text bar chart
    print('   {:<9} {:<9} {:>4.0f}%  {}'.format(part, PART_NUMBERS[part], p * 100, bar))

Spoken snag (transcribed):
   "Static on the radio, and the signal drops when the aircraft shakes."

Most likely part: Cable (P/N CBL-221)

Ranked shortlist (how sure, per part):
   Cable     CBL-221     42%  ############
   Antenna   ANT-4710    25%  ########
   HPF       HPF-2200    18%  #####
   LRU       LRU-118     15%  ####


> 🟩 **WHAT JUST HAPPENED**
>
> The full pipeline just ran end to end: a <b>spoken snag</b> became <b>text</b>, and that text became a <b>ranked shortlist of likely parts</b> &mdash; topped by the <b>Cable (CBL-221)</b>, with the antenna, filter and box close behind. Notice the modest confidence: a snag like "static, drops when it shakes" genuinely <i>could</i> be more than one part, so the model does not pretend to be certain. That is the honest answer &mdash; a prioritised starting point for the engineer, not a verdict.<br><br>And notice what it just handed you: the part number <b>CBL-221</b> &mdash; the very number we watched ASR mangle into CBL-227 a moment ago. The pilot described the snag in plain words; nobody had to read a fragile part number aloud.

## Step 6 · Try a different snag, to prove it is not a fluke

One example could be luck. Let us speak a *different* snag &mdash; a dead radio box this time &mdash; and run the whole pipeline again: make audio, transcribe, suggest the part. We bundle the three steps into one small helper so it is easy to reuse.

In [22]:
def identify_part(sentence):
    '''Speak the snag, transcribe it, and name the most likely part.'''
    # Step A: turn the sentence into audio (our stand-in for a recording).
    gTTS(sentence).save('clip.mp3')
    # Step B: transcribe the audio back into text.
    heard = model.transcribe('clip.mp3')['text'].strip()
    # Step C: embed the snag and ask the model which part it points to.
    v = embedder.encode([heard])
    part = classifier.predict(v)[0]
    confidence = classifier.predict_proba(v).max()
    print('Snag spoken :', sentence)
    print('Transcribed :', heard)
    print('Likely part :', part, '(P/N ' + PART_NUMBERS[part] + ')',
          ' -- {:.0f}% sure'.format(confidence * 100))
    return part

# A clearly different snag this time -- a dead box points at the LRU.
identify_part('The radio is completely dead with no power and the display is blank.')

/Users/aarshdesai/miniconda3/lib/python3.12/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Snag spoken : The radio is completely dead with no power and the display is blank.
Transcribed : The radio is completely dead with no power when the display is blank.
Likely part : LRU (P/N LRU-118)  -- 48% sure


np.str_('LRU')

> 🟩 **WHAT JUST HAPPENED**
>
> A completely different spoken snag ran through the same three steps and pointed at the <b>LRU</b> &mdash; the radio box. "Dead", "no power" and "display is blank" are the words the dead-box examples used, so the model leans there with more confidence than the noisier reception snags. The pipeline <code>speak &rarr; transcribe &rarr; suggest the part</code> is reusable for any snag &mdash; that one helper does it all.

> 🟧 **YOUR TURN**
>
> This is the most hands-on moment of the day &mdash; make the computer weigh up <b>your own</b> spoken snag.<br><br>In the cell just above, change the sentence inside <code>identify_part( ... )</code> to anything you like, then press <b>Shift + Enter</b>. Watch the suggested part change as the words change. A few to try:<br>1. <code>'The antenna is cracked and reception is poor in the rain.'</code> &rarr; should lean to the <b>Antenna</b>.<br>2. <code>'There is interference from the radar bleeding into com one.'</code> &rarr; should lean to the <b>HPF</b> filter.<br>3. Write one that mixes two parts' worth of clues (say, "interference AND the reception keeps dropping out") and watch the shortlist <b>split</b> &mdash; that is the computer telling you it is genuinely unsure, which is honest behaviour, not a failure. A snag rarely names one part on its own.

> 🟪 **WHAT WE LEARNED IN THIS NOTEBOOK**
>
> - <b>Speech analytics = speech-to-text + text analytics</b> &mdash; two familiar pieces snapped together.
> - <b>Text-to-speech</b> only made our demo clip; the real work is <b>speech-to-text (ASR)</b>, which turns audio into words.
> - Once speech becomes text, it is just text &mdash; so the <b>exact text-analytics method</b> weighs it up, no new ideas needed.
> - The showpiece pipeline: <b>speak &rarr; transcribe &rarr; suggest the likely part</b>, as a ranked shortlist with confidence.
> - <b>Be honest:</b> a snag rarely pins one part, so the model gives a prioritised shortlist &mdash; a starting point for the engineer, not the last word. And a low WER can still miss the one word that matters.

## Wrapping up the day

> 💡 **THE BIG IDEA**
>
> Step back and look at the pipeline you just ran: a <b>spoken snag</b> became <b>text</b>, and that text became a <b>ranked shortlist of likely parts</b> &mdash; the very same words-to-numbers method from the text sessions, with one new front end (listening). And you saw it <b>honestly</b>: speech-to-text is excellent on clear speech and measurable with <b>WER</b>, but a low error rate can still miss the one word that matters &mdash; which is why a small fixed vocabulary (keyword spotting) is so much more reliable when the stakes are high, and why critical part numbers always get a human's eyes.

**One recipe, three kinds of messy data &mdash; across the whole day:**

| Kind of data | Where it appeared | The one idea, every time |
|---|---|---|
| **Text** &mdash; written maintenance notes | Sessions 1 &amp; 2 (urgent vs routine, recurring themes) | learn the pattern from **past examples**, then make a call on a **new case** |
| **Sound** &mdash; a machine's vibration / audio | Session 3 (hear a failing machine) | *(same idea)* |
| **Speech** &mdash; a pilot's spoken snag | Session 4 &mdash; this session | *(same idea)* |

> 🟪 **COMING UP NEXT**
>
> <b>That is the whole day.</b> Across four sessions one idea kept returning: <b>turn it into numbers, learn the pattern, judge the new case</b> &mdash; and always check the result honestly. You classified written maintenance notes (<b>urgent vs routine</b>), found the <b>recurring themes</b> hiding in free text, taught a model to <b>hear a failing machine</b> from its sound alone, and finally turned a <b>spoken snag into text</b> and used it to suggest the <b>likely part</b>. Same recipe, four very different signals &mdash; neither magic nor nonsense, and now firmly in your hands.